In [ ]:
from function import *
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM, Dense
from keras.callbacks import TensorBoard
label_map = {label:num for num, label in enumerate(actions)}
# print(label_map)
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        print(action)
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action,str(sequence), "{}.npy".format(frame_num)),allow_pickle=True)
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

X = np.array(sequences)
y = to_categorical(labels).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30,63)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))
res = [.7, 0.2, 0.1]

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.fit(X_train, y_train, epochs=100, callbacks=[tb_callback])
model.summary()

model_json = model.to_json()
with open("model2.json", "w") as json_file:
    json_file.write(model_json)
model.save('model2.h5')

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
import os
import numpy as np
from function import *
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM, Dense
from keras.callbacks import TensorBoard
label_map = {label:num for num, label in enumerate(actions)}
# print(label_map)
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        try:
            res = np.load(os.path.join(DATA_PATH, action, "{}.npy".format(sequence)),allow_pickle=True)
            sequences.append(res)
            labels.append(label_map[action])
        except:
            print('')

X = np.array(sequences)
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

# ... (your previous code)

# Ensure log directory exists for TensorBoard
log_dir = os.path.join('Logs')
os.makedirs(log_dir, exist_ok=True)
tb_callback = TensorBoard(log_dir=log_dir)

# Create a Sequential model

model = Sequential([
    Flatten(input_shape=(21, 2), name='flatten_2'),  # Flattens input of shape (21, 2) to (None, 42)
    Dense(64, activation='relu', name='dense_11'),  # Dense layer with 64 units
    Dense(128, activation='relu', name='dense_12'),  # Dense layer with 128 units
    Dense(512, activation='relu', name='dense_13'),  # Dense layer with 512 units
    Dense(64, activation='relu', name='dense_14'),  # Dense layer with 64 units
    Dense(32, activation='relu', name='dense_15'),  # Dense layer with 32 units
    Dense(16, activation='softmax', name='dense_16')  # Dense layer with 10 units and softmax activation
])

# Compile the model and specify loss and optimizer
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Summary of the model architecture
model.summary()

# Train the model
model.fit(X_train, y_train, epochs=100, callbacks=[tb_callback], validation_data=(X_test, y_test))

# Save the model
model.save('model1')
